# 10 — A/B Test Simulation

**⚠️ This is a SIMULATION, not a real experiment.**

This notebook demonstrates the concept of using experimentation to validate
whether a model-based intervention actually improves delivery outcomes.

**Scenario:**
- **Control group**: Orders handled with the normal process (no model)
- **Treatment group**: High-risk orders (flagged by our model) receive
  proactive intervention (e.g., priority routing, backup rider assignment)

**Why this matters:**
Building a good model is only half the job. We need to know if ACTING on
the model's predictions actually improves outcomes. This is the kind of
thinking data scientists at companies like Swiggy use daily.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from src.evaluation import get_risk_category

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 10.1 Setup: Load Model Predictions

In [ ]:
# Load model and test data
model = joblib.load("../models/best_model.pkl")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

# Drop same high-cardinality columns as in training
high_card_cols = ["Order City", "Order State", "Customer City", "Customer State"]
X_test = X_test.drop(columns=[c for c in high_card_cols if c in X_test.columns])

y_prob = model.predict_proba(X_test)[:, 1]
risk_cats = get_risk_category(y_prob)

print(f"Test set size: {len(y_test):,}")
print(f"Actual late delivery rate: {y_test.mean()*100:.1f}%")
print(f"\nRisk categories:")
print(pd.Series(risk_cats).value_counts())


## 10.2 Simulate the Experiment

We simulate what would happen if we randomly split orders into control
and treatment groups, and then intervened on high-risk orders in the
treatment group.

**Assumptions (clearly stated):**
- Intervention reduces late delivery probability by 30% for high-risk orders
- Intervention has no effect on low/medium risk orders
- The 30% reduction is an assumption — in reality, you'd measure this

These numbers are made up for illustration. In a real company, you'd
run an actual experiment to measure the true effect.

In [ ]:
# Simulate: randomly assign to control/treatment
n = len(y_test)
assignment = np.random.choice(["Control", "Treatment"], size=n)

sim_df = pd.DataFrame({
    "group": assignment,
    "actual_late": y_test.values,
    "prob_late": y_prob,
    "risk_category": risk_cats
})

# In the treatment group, simulate intervention effect on high-risk orders
# Assumption: intervention reduces late delivery probability by 30%
INTERVENTION_EFFECT = 0.30

sim_df["simulated_late"] = sim_df["actual_late"].copy()

# For treatment + high risk: flip some late deliveries to on-time
treatment_high_risk = (
    (sim_df["group"] == "Treatment") & 
    (sim_df["risk_category"] == "High Risk") &
    (sim_df["actual_late"] == 1)
)

n_flipped = int(treatment_high_risk.sum() * INTERVENTION_EFFECT)
flip_indices = sim_df[treatment_high_risk].sample(n=n_flipped, random_state=42).index
sim_df.loc[flip_indices, "simulated_late"] = 0

print(f"Orders in simulation: {n:,}")
print(f"Control group: {(sim_df['group']=='Control').sum():,}")
print(f"Treatment group: {(sim_df['group']=='Treatment').sum():,}")
print(f"High-risk orders intervened: {treatment_high_risk.sum():,}")
print(f"Late deliveries flipped to on-time: {n_flipped:,}")

## 10.3 Compare Groups

In [ ]:
control = sim_df[sim_df["group"] == "Control"]
treatment = sim_df[sim_df["group"] == "Treatment"]

control_rate = control["simulated_late"].mean()
treatment_rate = treatment["simulated_late"].mean()

print("Results:")
print("=" * 50)
print(f"Control late delivery rate:   {control_rate*100:.2f}%")
print(f"Treatment late delivery rate: {treatment_rate*100:.2f}%")
print(f"Absolute difference:          {(control_rate - treatment_rate)*100:.2f} percentage points")
if control_rate > 0:
    print(f"Relative reduction:           {((control_rate - treatment_rate)/control_rate)*100:.1f}%")

## 10.4 Statistical Significance

Is the difference between control and treatment statistically significant?

- **H₀:** Late delivery rate is the same in both groups
- **H₁:** Treatment group has a lower late delivery rate
- **Test:** Two-proportion z-test
- **Significance level:** α = 0.05

In [ ]:
# Two-proportion z-test
n_control = len(control)
n_treatment = len(treatment)
p_control = control_rate
p_treatment = treatment_rate

# Pooled proportion
p_pooled = (control["simulated_late"].sum() + treatment["simulated_late"].sum()) / (n_control + n_treatment)

# Z-statistic
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_control + 1/n_treatment))
if se > 0:
    z_stat = (p_control - p_treatment) / se
    p_value = 1 - stats.norm.cdf(z_stat)  # One-sided test
else:
    z_stat = 0
    p_value = 1

print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value (one-sided): {p_value:.6f}")
print()
if p_value < 0.05:
    print("✓ Result is statistically significant at α=0.05")
    print("  The model-based intervention appears to reduce late deliveries.")
else:
    print("✗ Result is NOT statistically significant")
    print("  We cannot conclude the intervention had an effect.")

# Confidence interval for the difference
se_diff = np.sqrt(p_control*(1-p_control)/n_control + p_treatment*(1-p_treatment)/n_treatment)
diff = p_control - p_treatment
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"\n95% CI for difference: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

## 10.5 Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot of rates
rates = pd.DataFrame({
    "Group": ["Control", "Treatment"],
    "Late Delivery Rate": [control_rate, treatment_rate]
})
axes[0].bar(rates["Group"], rates["Late Delivery Rate"], 
            color=["#e74c3c", "#27ae60"], alpha=0.7)
axes[0].set_ylabel("Late Delivery Rate")
axes[0].set_title("Control vs Treatment — Late Delivery Rate")
for i, (_, row) in enumerate(rates.iterrows()):
    axes[0].text(i, row["Late Delivery Rate"] + 0.005, 
                 f'{row["Late Delivery Rate"]*100:.2f}%', ha="center")

# Breakdown by risk category
for i, group in enumerate(["Control", "Treatment"]):
    subset = sim_df[sim_df["group"] == group]
    risk_rates = subset.groupby("risk_category")["simulated_late"].mean()
    risk_rates = risk_rates.reindex(["Low Risk", "Medium Risk", "High Risk"])
    
    x_pos = np.arange(3) + i * 0.35
    axes[1].bar(x_pos, risk_rates.values, width=0.3, 
                label=group, alpha=0.7,
                color=["#e74c3c", "#27ae60"][i])

axes[1].set_xticks(np.arange(3) + 0.175)
axes[1].set_xticklabels(["Low Risk", "Medium Risk", "High Risk"])
axes[1].set_ylabel("Late Delivery Rate")
axes[1].set_title("Late Rate by Risk Category and Group")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

**What this simulation demonstrates:**

1. **Experiment design**: How to structure a control/treatment comparison
   for evaluating model-based interventions

2. **Statistical testing**: Using a z-test to determine if the observed
   difference is statistically significant

3. **Confidence intervals**: Quantifying the uncertainty around the
   measured effect

**Important caveats:**
- This is a **simulation**, not a real experiment
- The 30% intervention effect is an **assumption**, not a measured quantity
- In a real company (like Swiggy), you would:
  - Run this as a proper randomized controlled trial
  - Use the actual delivery outcomes
  - Account for network effects and other confounders
  - Run the experiment for a sufficient duration

**Transferable skills:**
- Understanding A/B testing methodology
- Knowing how to measure if a model creates real business value
- Awareness of statistical significance vs practical significance
- This kind of experimentation is central to data science at consumer-tech companies